# Model scaling and architecture analysis

Purpose: examine whether nominal parameter count, active MoE parameters, architecture, model family, and quantization explain differences in inference speed.

This notebook reads only `results/processed/final-analysis-dataset.csv` and writes derived figures and tables under `analysis/`. Missing measurements are excluded from the relevant calculation rather than replaced with zero.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import load_dataset, save_figure, save_table, successful, grouped_bar, add_display_hardware

data = load_dataset(PROJECT_ROOT / "results" / "processed" / "final-analysis-dataset.csv")
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")

## Model characteristics

The characteristics table provides one metadata record per model and makes the parameter and quantization assumptions auditable.

In [ ]:
model_characteristics = (data.groupby("model", as_index=False).agg(model_size_b=("model_size_b", "first"), active_parameters_b=("active_parameters_b", "first"), architecture=("architecture", "first"), model_family=("model_family", "first"), quantization=("quantization", "first"), workloads=("workload", "nunique"), hardware=("hardware", "nunique")))
display(model_characteristics)
save_table(model_characteristics, "04_model_characteristics.csv")

## Dense versus MoE models

Architecture-level means compare speed, latency, and memory while retaining the architecture label. Results are descriptive and should not be interpreted as causal without controlling for hardware and workload.

In [ ]:
successful_data = successful(data)
architecture_comparison = successful_data.groupby("architecture", as_index=False).agg(models=("model", "nunique"), decode_tps=("decode_tps", "mean"), prompt_eval_tps=("prompt_eval_tps", "mean"), total_time=("total_time", "mean"), ram_usage=("ram_usage", "mean"), vram_usage=("vram_usage", "mean"), overall_score=("overall_score", "mean"))
display(architecture_comparison)
save_table(architecture_comparison, "04_architecture_comparison.csv")
fig, ax = plt.subplots(figsize=(8, 5))
architecture_comparison.set_index("architecture")[["decode_tps", "prompt_eval_tps"]].plot(kind="bar", ax=ax)
ax.set_title("Dense and MoE mean inference speed")
ax.set_xlabel("Architecture")
ax.set_ylabel("Mean throughput (tokens/s)")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "04_architecture_speed.png")
plt.show()

## Total parameters versus inference speed

This plot uses model-level mean decode throughput across successful observations. Hardware and workload variation is not removed, so the plot is a scaling overview rather than a controlled benchmark.

In [ ]:
model_speed = successful_data.groupby(["model", "model_size_b", "architecture"], as_index=False).agg(decode_tps=("decode_tps", "mean"))
fig, ax = plt.subplots(figsize=(9, 6))
for architecture, subset in model_speed.groupby("architecture"):
    ax.scatter(subset["model_size_b"], subset["decode_tps"], label=architecture, s=70)
    for _, row in subset.iterrows(): ax.annotate(row["model"], (row["model_size_b"], row["decode_tps"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Total model parameters versus inference speed")
ax.set_xlabel("Model size (B parameters)")
ax.set_ylabel("Mean decode throughput (tokens/s)")
ax.legend(title="Architecture")
ax.grid(alpha=0.25)
save_figure(fig, "04_total_parameters_vs_speed.png")
plt.show()
save_table(model_speed, "04_model_speed_summary.csv")

## Active parameters versus MoE inference speed

For MoE models, active parameters better represent the computation used per token than total parameter count. Only MoE rows with active-parameter metadata are plotted.

In [ ]:
moe_speed = successful_data.loc[successful_data["architecture"].eq("MoE")].groupby(["model", "active_parameters_b", "model_size_b"], as_index=False).agg(decode_tps=("decode_tps", "mean"))
display(moe_speed)
save_table(moe_speed, "04_moe_active_parameters_speed.csv")
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(moe_speed["active_parameters_b"], moe_speed["decode_tps"], s=80)
for _, row in moe_speed.iterrows(): ax.annotate(row["model"], (row["active_parameters_b"], row["decode_tps"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("Active MoE parameters versus inference speed")
ax.set_xlabel("Active parameters (B)")
ax.set_ylabel("Mean decode throughput (tokens/s)")
ax.grid(alpha=0.25)
save_figure(fig, "04_active_parameters_vs_speed.png")
plt.show()

## Quantization impact

Quantization groups are compared descriptively. Since quantization is confounded with model and architecture in this dataset, the table should be used as contextual evidence rather than an isolated causal estimate.

In [ ]:
quantization_comparison = successful_data.groupby("quantization", as_index=False).agg(models=("model", "nunique"), decode_tps=("decode_tps", "mean"), total_time=("total_time", "mean"), ram_usage=("ram_usage", "mean"), vram_usage=("vram_usage", "mean"))
display(quantization_comparison)
save_table(quantization_comparison, "04_quantization_comparison.csv")
fig, ax = plt.subplots(figsize=(8, 5))
quantization_comparison.set_index("quantization")["decode_tps"].plot(kind="bar", ax=ax)
ax.set_title("Mean decode throughput by quantization")
ax.set_xlabel("Quantization")
ax.set_ylabel("Mean decode throughput (tokens/s)")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", alpha=0.25)
save_figure(fig, "04_quantization_speed.png")
plt.show()